# Guyton ve Hall - Colab Pro+ Surya OCR
Soru Çözücü kitap hafızası için tüm sayfaları, Mac'e atomik kontrol noktası göndererek işler. Kesilirse aynı hücre yeniden çalıştırıldığında kaldığı yerden sürer.

In [ ]:
PDF_URL = 'PDF_URL_BURAYA'
CHECKPOINT_URL = 'CHECKPOINT_URL_BURAYA'

In [ ]:
!pip -q install 'surya-ocr==0.22.1' 'pypdfium2>=5.10.1'

In [ ]:
import os, subprocess, torch
assert torch.cuda.is_available(), 'GPU çalışma zamanı seçilmedi'
gpu = torch.cuda.get_device_properties(0)
vram_gb = round(gpu.total_memory / 2**30)
parallel = 8 if vram_gb <= 24 else (16 if vram_gb <= 48 else 24)
print(gpu.name, vram_gb, 'GB', 'parallel', parallel)
if not os.path.exists('/content/llama.cpp/.git'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', '/content/llama.cpp'])
subprocess.check_call(['cmake', '-S', '/content/llama.cpp', '-B', '/content/llama.cpp/build', '-DGGML_CUDA=ON', '-DLLAMA_CURL=OFF', '-DCMAKE_BUILD_TYPE=Release'])
subprocess.check_call(['cmake', '--build', '/content/llama.cpp/build', '--config', 'Release', '--target', 'llama-server', '-j', '2'])
llama_server = '/content/llama.cpp/build/bin/llama-server'
assert os.path.isfile(llama_server), llama_server
os.environ['LLAMA_CPP_BINARY'] = llama_server
os.environ['SURYA_INFERENCE_BACKEND'] = 'llamacpp'
os.environ['SURYA_INFERENCE_PARALLEL'] = str(parallel)
os.environ['SURYA_INFERENCE_CTX_SIZE'] = str(max(16384, parallel * 12288))
os.environ['SURYA_INFERENCE_KEEP_ALIVE'] = 'true'
print('Surya CUDA llama.cpp hazır')

In [ ]:
import html, json, re, shutil, time, urllib.error, urllib.request
from pathlib import Path
import pypdfium2
from surya.recognition import RecognitionPredictor
from google.colab import files

PDF = Path('/content/guyton.pdf')
OUT = Path('/content/guyton_ocr.jsonl')
urllib.request.urlretrieve(PDF_URL, PDF)
try:
    urllib.request.urlretrieve(CHECKPOINT_URL, OUT)
    print('Mac kontrol noktası alındı:', OUT.stat().st_size, 'bayt')
except urllib.error.HTTPError as exc:
    if exc.code != 404: raise

pdf = pypdfium2.PdfDocument(PDF)
expected = list(range(1, len(pdf) + 1))
done = set()
if OUT.exists():
    for line in OUT.read_text(encoding='utf-8').splitlines():
        try: done.add(int(json.loads(line)['page']))
        except Exception: pass
print('PDF sayfası:', len(pdf), '· devam noktası:', len(done), '/', len(expected))

predictor = RecognitionPredictor()
batch_size = 8 if vram_gb <= 24 else (16 if vram_gb <= 48 else 24)
tag = re.compile(r'<[^>]+>')
breaks = re.compile(r'</(?:p|div|li|tr|h\d)>|<br\s*/?>', re.I)

def plain(page):
    blocks = sorted(getattr(page, 'blocks', None) or [], key=lambda b: getattr(b, 'reading_order', 0) or 0)
    texts, confs = [], []
    for block in blocks:
        if getattr(block, 'skipped', False) or getattr(block, 'error', None): continue
        text = html.unescape(tag.sub('', breaks.sub('\n', getattr(block, 'html', '') or '')))
        text = '\n'.join(x.strip() for x in text.splitlines() if x.strip())
        if text:
            texts.append(text); confs.append(float(getattr(block, 'confidence', 0) or 0))
    return '\n'.join(texts), (sum(confs) / len(confs) if confs else 0.0)

def append_record(record):
    with OUT.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n'); f.flush()

def push_checkpoint():
    request = urllib.request.Request(CHECKPOINT_URL, data=OUT.read_bytes(), method='POST', headers={'Content-Type': 'application/x-ndjson'})
    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(request, timeout=180) as response:
                status = json.loads(response.read().decode())
                print('Mac checkpoint', status['pages'], 'sayfa'); return
        except Exception as exc:
            print('checkpoint yeniden deneme', attempt, type(exc).__name__, exc); time.sleep(10 * attempt)
    raise RuntimeError('Mac checkpoint gönderilemedi')

remaining = [n for n in expected if n not in done]
for start in range(0, len(remaining), batch_size):
    nums = remaining[start:start + batch_size]
    images = [pdf[n-1].render(scale=300/72).to_pil() for n in nums]
    result = None
    for attempt in range(1, 4):
        try:
            result = predictor(images); break
        except Exception as exc:
            print('yeniden deneme', nums, attempt, type(exc).__name__, exc); time.sleep(10 * attempt)
    if result is None:
        for n, image in zip(nums, images):
            try:
                one = predictor([image])[0]; text, confidence = plain(one)
                append_record({'page': n, 'text': text, 'confidence': confidence, 'error': ''})
            except Exception as exc:
                append_record({'page': n, 'text': '', 'confidence': 0.0, 'error': f'{type(exc).__name__}: {exc}'})
    else:
        for n, page in zip(nums, result):
            text, confidence = plain(page)
            append_record({'page': n, 'text': text, 'confidence': confidence, 'error': ''})
    push_checkpoint()
    print(f'{min(start + len(nums), len(remaining))}/{len(remaining)} · son sayfa {nums[-1]}')
pdf.close()

records = [json.loads(x) for x in OUT.read_text(encoding='utf-8').splitlines() if x.strip()]
pages = {int(x['page']) for x in records}
missing = sorted(set(expected) - pages)
errors = [x for x in records if x.get('error')]
print('kayıt', len(pages), 'eksik', len(missing), 'hatalı', len(errors))
assert not missing and not errors, (missing[:20], errors[:3])
shutil.make_archive('/content/guyton_colab_result', 'zip', '/content', 'guyton_ocr.jsonl')
files.download('/content/guyton_colab_result.zip')